# GEPA prompt optimization for 8 local datasets

This notebook runs `dspy.teleprompt.GEPA` over local datasets located in
[`datasets/`](datasets), using **gpt-5-nano** as both the task LM and the
reflection LM (temperature 0.7).

Three optimization metrics are supported: **bert_score**, **exact_match**, **f1**.
The metric is auto-selected per dataset (or can be overridden via `metric_name=...`):
- `gsm8k`      → `exact_match`
- `tweeteval`  → `f1`
- all others   → `bert_score`

Supported datasets:
`squad_v2`, `gsm8k`, `common_gen`, `xsum`, `tweeteval`, `mediqa`, `code_to_text`, `concode`.

Hyper-parameters (fixed per task spec):
- population size (`reflection_minibatch_size`) = 10
- number of iterations / epochs (`max_full_evals`) = 5
- splits train/val/test = 50/100/300

For every dataset we save
`results/<dataset>.json` containing:
- the initial prompt
- the per-iteration optimization trace (candidate prompts + scores)
- the final optimized prompt

In [ ]:
%pip install -q dspy-ai bert-score langchain-openai

In [ ]:
import os
import json
from getpass import getpass
from pathlib import Path

import utils
from utils import (
    DATASETS,
    METRICS,
    DATASET_DEFAULT_METRIC,
    DEFAULT_HPARAMS,
    run_optimization,
    configure_lms,
)

print('Available datasets:', DATASETS)
print('Available metrics :', METRICS)
print('Per-dataset default metric:')
for d, m in DATASET_DEFAULT_METRIC.items():
    print(f'  {d:15s} -> {m}')
print('Default hyper-parameters:')
for k, v in DEFAULT_HPARAMS.items():
    print(f'  {k}: {v}')

In [ ]:
# Put your OpenAI key in the environment (or paste it directly here).
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("Enter your OpenAI API key: ")
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

## Optimize a single dataset

Change `DATASET_NAME` to one of the supported datasets.  
You can also override any hyper-parameter via keyword arguments to `run_optimization`.

In [ ]:
# # === pick the dataset to optimize on ===
# DATASET_NAME = 'gsm8k'  # one of utils.DATASETS
#
# artifact = run_optimization(
#     dataset_name=DATASET_NAME,
#     openai_api_key=OPENAI_API_KEY,
#     model=DEFAULT_HPARAMS['model'],                      # 'openai/gpt-5-nano'
#     temperature=DEFAULT_HPARAMS['temperature'],          # 0.7
#     max_tokens=DEFAULT_HPARAMS['max_tokens'],
#     train_n=DEFAULT_HPARAMS['train_n'],                  # 50
#     val_n=DEFAULT_HPARAMS['val_n'],                      # 100
#     test_n=DEFAULT_HPARAMS['test_n'],                    # 300
#     population_size=DEFAULT_HPARAMS['population_size'],  # 10
#     num_iterations=DEFAULT_HPARAMS['num_iterations'],    # 5
#     num_threads=DEFAULT_HPARAMS['num_threads'],
#     candidate_selection_strategy=DEFAULT_HPARAMS['candidate_selection_strategy'],
#     use_merge=DEFAULT_HPARAMS['use_merge'],
#     max_merge_invocations=DEFAULT_HPARAMS['max_merge_invocations'],
#     skip_perfect_score=DEFAULT_HPARAMS['skip_perfect_score'],
#     perfect_score=DEFAULT_HPARAMS['perfect_score'],
#     failure_score=DEFAULT_HPARAMS['failure_score'],
#     seed=DEFAULT_HPARAMS['seed'],
#     results_dir='results',
#     log_root='runs',
#     evaluate_on_test=False,
# )
#
# print('\n--- INITIAL PROMPT ---')
# print(artifact['initial_prompt'])
# print('\n--- FINAL PROMPT ---')
# print(artifact['final_prompt'])

## Sequential optimization for all 8 datasets (explicit hyper-parameters)

Runs GEPA sequentially over every dataset with:
- model `openai/gpt-5-nano`, temperature `1.0`
- population size (`reflection_minibatch_size`) = 10
- iterations / epochs (`max_full_evals`) = 5
- train/val/test = 50/100/300
- per-dataset metric: `gsm8k`→`exact_match`, `tweeteval`→`f1`, others→`bert_score`

Saves `results/<dataset>_<metric>.json` for each dataset, plus a combined `results/_summary.json`.

In [ ]:
import torch
import gc
import warnings

warnings.filterwarnings('ignore')

torch.cuda.empty_cache()
gc.collect()

In [ ]:
from utils import DATASETS, run_optimization

# Per-dataset metric assignment.
DATASET_TO_METRIC = {
    'gsm8k': 'exact_match',
    'tweeteval': 'f1',
    'squad_v2': 'bert_score',
    'common_gen': 'bert_score',
    'xsum': 'bert_score',
    'mediqa': 'bert_score',
    'code_to_text': 'bert_score',
    'concode': 'bert_score',
}

# GEPA hyper-parameters shared across every dataset.
COMMON_KWARGS = dict(
    openai_api_key=OPENAI_API_KEY,
    model='openai/gpt-5-nano',
    temperature=1.0,
    max_tokens=16000,
    train_n=50,
    val_n=100,
    test_n=300,
    population_size=10,                       # GEPA: reflection_minibatch_size
    num_iterations=5,                         # GEPA: max_full_evals
    num_threads=8,
    candidate_selection_strategy='pareto',
    use_merge=True,
    max_merge_invocations=5,
    skip_perfect_score=True,
    perfect_score=1.0,
    failure_score=0.0,
    seed=0,
    results_dir='results',
    log_root='runs',
    evaluate_on_test=False,
)

summary = {}
for name in DATASETS:
    metric = DATASET_TO_METRIC[name]
    print(f'\n========== {name}  (metric: {metric}) ==========')
    try:
        art = run_optimization(
            dataset_name=name,
            metric_name=metric,
            **COMMON_KWARGS,
        )
        summary[name] = {
            'dataset': name,
            'metric': art['metric'],
            'initial_prompt': art['initial_prompt'],
            'final_prompt': art['final_prompt'],
        }
        print(f'[{name}] done -> results/{name}_{metric}.json')
    except Exception as exc:
        print(f'[{name}] FAILED: {exc}')
        summary[name] = {'dataset': name, 'metric': metric, 'error': str(exc)}

Path('results').mkdir(parents=True, exist_ok=True)
with open('results/_summary.json', 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)

print('\nAll datasets processed. Artifacts: ./results/<dataset>_<metric>.json')